# FNO3D — Physics-Constrained Fourier Neural Operator (3D)

Reconstruit un volume atmospherique 3D (u, v, w, rho, T, RH, p) de 0 a 1000m ASL a partir
des mesures ANEO, en suivant la specification "Approche de Reconstruction Atmospherique 3D"
(R. Ben Amor, 25/01/2026) : les capteurs ANEO ne sont PAS coplanaires
(z_i = z_terrain(x_i,y_i) + z_rel_i), donc le reseau echantillonne un volume 3D irregulier,
pas un plan. Ce notebook est independant de `fno2D.ipynb` (qui reste la baseline 2D de
comparaison) — chemins, tenseurs et checkpoints separes, aucun fichier partage.

**Grille verticale** : 7 niveaux ASL (metres), meme convention que `ANEO_FNO2D_Model.ipynb` :
`[25, 50, 100, 200, 300, 500, 1000]`. Verifie empiriquement sur `aneo_grid_points.csv`
(500 000 points, z_abs = z_terrain+z_rel dans [2m, 859m]) : les niveaux 25-500m ont une
couverture reelle de 7-28% des colonnes chacun ; le niveau 1000m n'a quasiment aucune
observation directe (0.02%, 116 points) et sert uniquement de condition aux limites physique
(gradient vertical faible en altitude, cf. specification section 4.4).

**Grille horizontale** : sous-echantillonnee d'un facteur 2 (1000x500 -> 500x250) pour que
l'ajout de l'axe z tienne en memoire GPU — FNO2D tourne deja a la limite (batch=1,
accumulation de gradient, precision mixte) sur grille pleine resolution sans axe z.

**Masquage** : chaque voxel (z,y,x) sous le terrain local est non-physique ("souterrain") et
exclu de toutes les losses via `valid_mask`. Chaque snapshot ne fournit une vraie observation
qu'a UN SEUL niveau z par colonne (celui le plus proche de z_terrain+z_rel a ce point) — la
loss de donnees est donc masquee par `obs_mask`, le reste du volume n'etant contraint que par
la physique (conservation de masse + conditions aux limites).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device} | ✓ Imports OK")

In [ ]:
DRIVE_DIR = Path("/content/drive/MyDrive")
BASE_DIR  = Path("/content/aneo_work_3d")

DATA_DIR        = DRIVE_DIR / "data_4x"
OUT_DIR         = BASE_DIR  / "aneo_output_3d"
MODEL_DIR       = BASE_DIR  / "aneo_models_3d"
TENSOR_DIR      = BASE_DIR  / "aneo_tensors_3d"
FIG_DIR         = BASE_DIR  / "aneo_figures_3d"
DRIVE_MODEL_DIR = DATA_DIR  / "models_3d"   # persiste malgre les deconnexions Colab

for d in [OUT_DIR, MODEL_DIR, TENSOR_DIR, FIG_DIR, DRIVE_MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"✓ {d}")

print(f"\nAcces Drive source :")
print(f"  data_4x existe : {DATA_DIR.exists()}")
print(f"  aneo_grid_points.csv : {(DATA_DIR/'aneo_grid_points.csv').exists()}")

In [ ]:
NX_FULL, NY_FULL = 1000, 500
DOWNSAMPLE = 2
NX, NY = NX_FULL // DOWNSAMPLE, NY_FULL // DOWNSAMPLE

Z_LEVELS = np.array([25, 50, 100, 200, 300, 500, 1000], dtype=np.float32)
NZ = len(Z_LEVELS)
Z_EDGES = np.concatenate([[0.0], (Z_LEVELS[:-1] + Z_LEVELS[1:]) / 2, [np.inf]]).astype(np.float32)

GRID3D_PATH = TENSOR_DIR / "grid3d_static.npz"


def normalize(arr):
    return (arr - arr.mean()) / (arr.std() + 1e-8)


if GRID3D_PATH.exists():
    print("✓ grid3d_static.npz existe deja")
    g3d = np.load(GRID3D_PATH)
    z_terrain_ds    = g3d['z_terrain']
    valid_mask      = g3d['valid_mask']
    height_agl_norm = g3d['height_agl_norm']
    static_features = g3d['static_features']
else:
    grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv")
    grid = grid.sort_values(['iy', 'ix']).reset_index(drop=True)
    assert len(grid) == NX_FULL * NY_FULL

    def to_grid(col):
        return grid[col].values.reshape(NY_FULL, NX_FULL)

    z_terrain_full = to_grid('z_terrain_m').astype(np.float32)
    z0_full        = to_grid('z0_m').astype(np.float32)
    slope_full     = to_grid('slope_deg').astype(np.float32)
    aspect_full    = to_grid('aspect_deg').astype(np.float32)
    x_full         = to_grid('x_m').astype(np.float32)
    y_full         = to_grid('y_m').astype(np.float32)

    # sous-echantillonnage horizontal par striding (grille reguliere, pas d'interpolation)
    z_terrain_ds = z_terrain_full[::DOWNSAMPLE, ::DOWNSAMPLE]
    z0_ds        = z0_full[::DOWNSAMPLE, ::DOWNSAMPLE]
    slope_ds     = slope_full[::DOWNSAMPLE, ::DOWNSAMPLE]
    aspect_ds    = aspect_full[::DOWNSAMPLE, ::DOWNSAMPLE]
    x_ds         = x_full[::DOWNSAMPLE, ::DOWNSAMPLE]
    y_ds         = y_full[::DOWNSAMPLE, ::DOWNSAMPLE]
    assert z_terrain_ds.shape == (NY, NX)

    # valid_mask[z,y,x] = 1 si le niveau ASL est au-dessus du sol local, 0 sinon (souterrain)
    valid_mask = (Z_LEVELS[:, None, None] >= z_terrain_ds[None, :, :]).astype(np.float32)

    # hauteur reelle au-dessus du sol par voxel (m) — signal cle pour que le reseau
    # sache "a quel niveau" il predit (equivalent du canal temporel dans FNO2D)
    height_agl = np.clip(Z_LEVELS[:, None, None] - z_terrain_ds[None, :, :], 0, None)
    height_agl_norm = normalize(height_agl).astype(np.float32)

    static_features = np.stack([
        normalize(z_terrain_ds), normalize(z0_ds), normalize(slope_ds),
        np.sin(np.deg2rad(aspect_ds)), np.cos(np.deg2rad(aspect_ds)),
        normalize(x_ds), normalize(y_ds),
    ], axis=0).astype(np.float32)   # (7, NY, NX) — statique, pas de dependance en z

    np.savez(GRID3D_PATH, z_terrain=z_terrain_ds, valid_mask=valid_mask,
             height_agl_norm=height_agl_norm, static_features=static_features)
    print(f"✓ grid3d_static.npz sauvegarde")

print(f"  Grille : {NX}x{NY} horizontal (downsample x{DOWNSAMPLE}) x {NZ} niveaux ASL")
for i, z in enumerate(Z_LEVELS):
    print(f"    niveau {z:>6.0f}m : {100*valid_mask[i].mean():5.1f}% de la surface au-dessus du sol")

In [ ]:
if (TENSOR_DIR / "split.json").exists():
    print("✓ Tenseurs 3D deja crees — cellule ignoree")
    print(f"  Fichiers _X.npy : {len(list(TENSOR_DIR.glob('*_X.npy')))}")
else:
    print("Creation des tenseurs 3D depuis les CSV...")

    print("\nEtape 1/5 — Grid points + niveau ASL par point + sous-echantillonnage horizontal...")
    grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv")
    grid = grid.sort_values(['iy', 'ix']).reset_index(drop=True)
    assert len(grid) == NX_FULL * NY_FULL

    keep = (grid['ix'] % DOWNSAMPLE == 0) & (grid['iy'] % DOWNSAMPLE == 0)
    grid_ds = grid[keep].copy()
    grid_ds['ix_ds'] = (grid_ds['ix'] // DOWNSAMPLE).astype(int)
    grid_ds['iy_ds'] = (grid_ds['iy'] // DOWNSAMPLE).astype(int)
    assert len(grid_ds) == NX * NY

    # z_i = z_terrain(x_i,y_i) + z_rel_i (hauteur reelle du capteur, PAS une hauteur AGL fixe)
    z_abs = (grid_ds['z_terrain_m'] + grid_ds['z_rel_m']).values.astype(np.float32)
    z_idx = np.clip(np.digitize(z_abs, Z_EDGES) - 1, 0, NZ - 1)
    grid_ds['z_idx'] = z_idx

    land_use_vals = sorted(grid_ds['land_use_code'].dropna().unique())
    land_use_map  = {v: i for i, v in enumerate(land_use_vals)}
    grid_ds['land_use_enc'] = grid_ds['land_use_code'].map(land_use_map).fillna(0).astype(np.float32)
    lu_mean = float(grid_ds['land_use_enc'].mean())
    lu_std  = float(grid_ds['land_use_enc'].std()) + 1e-8
    grid_ds['land_use_enc_norm'] = (grid_ds['land_use_enc'] - lu_mean) / lu_std

    print(f"  Grid sous-echantillonne : {len(grid_ds):,} points ({NY}x{NX})")
    for i, z in enumerate(Z_LEVELS):
        pct = 100 * (grid_ds['z_idx'] == i).mean()
        print(f"    niveau {z:>6.0f}m : {pct:5.1f}% des colonnes y sont observees")

    print("\nEtape 2/5 — Identification des fichiers...")
    obs_files = sorted(DATA_DIR.glob("aneo_observations_day_*.csv"))
    print(f"  Fichiers observations : {len(obs_files)}")

    OBS_COLS = [
        'point_id', 'day_index', 'hour_utc',
        'temperature_c', 'air_density_kg_m3',
        'relative_humidity_pct', 'pressure_pa',
        'wind_u_m_s', 'wind_v_m_s', 'wind_w_m_s',
    ]
    # u,v,w en premier : la loss physique indexe les canaux 0/1/2 (u,v,w) et 3 (rho)
    TARGET_COLS = [
        'wind_u_m_s', 'wind_v_m_s', 'wind_w_m_s',
        'air_density_kg_m3', 'temperature_c',
        'relative_humidity_pct', 'pressure_pa',
    ]
    # directive de perimetre (Rayen, 2026-07-17) : tout ce qui est mesure sert
    # a la fois d'input (au niveau observe) et de target (sur tout le volume)
    NUMERIC_INPUT_COLS = TARGET_COLS
    DAY_COL, HOUR_COL = 'day_index', 'hour_utc'
    N_INPUT_CH = len(NUMERIC_INPUT_COLS) + 1   # + land_use

    print("\nEtape 3/5 — Calcul des stats de normalisation...")
    sum_dict  = {c: 0.0 for c in NUMERIC_INPUT_COLS}
    sum2_dict = {c: 0.0 for c in NUMERIC_INPUT_COLS}
    count = 0
    days_set, hours_set = set(), set()

    for i, f in enumerate(obs_files):
        df = pd.read_csv(f, usecols=OBS_COLS)
        df = df.dropna(subset=[DAY_COL, HOUR_COL])
        for col in NUMERIC_INPUT_COLS:
            vals = df[col].values.astype(np.float64)
            sum_dict[col]  += vals.sum()
            sum2_dict[col] += (vals**2).sum()
        days_set.update(df[DAY_COL].unique())
        hours_set.update(df[HOUR_COL].unique())
        count += len(df)
        if (i+1) % 10 == 0:
            print(f"  {i+1}/{len(obs_files)} fichiers analyses...")

    mean_dict = {c: sum_dict[c] / count for c in sum_dict}
    std_dict  = {
        c: float(np.sqrt(max(sum2_dict[c]/count - mean_dict[c]**2, 0) + 1e-8))
        for c in sum_dict
    }
    days  = sorted(int(d) for d in days_set)
    hours = sorted(int(h) for h in hours_set)
    print(f"  Jours  : {days}")
    print(f"  Heures : {hours}")

    norm_df = pd.DataFrame({c: {'mean': mean_dict[c], 'std': std_dict[c]} for c in TARGET_COLS}).T
    norm_df.to_csv(OUT_DIR / "normalization_stats.csv")

    print("\nEtape 4/5 — Creation des tenseurs .npy (X 2D, Y 3D, obs mask 3D)...")
    n_created = 0
    grid_indexed = grid_ds.set_index('point_id')

    for i, f in enumerate(obs_files):
        df = pd.read_csv(f, usecols=OBS_COLS)
        df = df.dropna(subset=[DAY_COL, HOUR_COL])
        df[DAY_COL]  = df[DAY_COL].astype(int)
        df[HOUR_COL] = df[HOUR_COL].astype(int)

        # ne garder que les points du grid sous-echantillonne (inner join)
        df = df.merge(
            grid_indexed[['ix_ds', 'iy_ds', 'z_idx', 'land_use_enc_norm']],
            left_on='point_id', right_index=True, how='inner')

        for col in NUMERIC_INPUT_COLS:
            df[col + '_norm'] = ((df[col] - mean_dict[col]) / std_dict[col]).astype(np.float32)

        input_norm_cols  = [c + '_norm' for c in NUMERIC_INPUT_COLS] + ['land_use_enc_norm']
        target_norm_cols = [c + '_norm' for c in TARGET_COLS]

        for day in df[DAY_COL].unique():
            for hour in df[HOUR_COL].unique():
                snap = df[(df[DAY_COL]==day) & (df[HOUR_COL]==hour)].sort_values(['iy_ds', 'ix_ds'])

                x_path = TENSOR_DIR / f"day_{day:03d}_h{hour:02d}_X.npy"
                y_path = TENSOR_DIR / f"day_{day:03d}_h{hour:02d}_Y.npy"
                m_path = TENSOR_DIR / f"day_{day:03d}_h{hour:02d}_M.npy"
                if x_path.exists():
                    n_created += 1
                    continue
                if len(snap) != NX * NY:
                    continue

                X = snap[input_norm_cols].values.reshape(NY, NX, N_INPUT_CH).astype(np.float32)

                Y = np.zeros((NY, NX, NZ, len(TARGET_COLS)), dtype=np.float32)
                M = np.zeros((NY, NX, NZ), dtype=np.float32)

                iy_arr = snap['iy_ds'].values
                ix_arr = snap['ix_ds'].values
                z_arr  = snap['z_idx'].values
                targ   = snap[target_norm_cols].values.astype(np.float32)

                Y[iy_arr, ix_arr, z_arr, :] = targ
                M[iy_arr, ix_arr, z_arr]    = 1.0

                np.save(x_path, X)
                np.save(y_path, Y)
                np.save(m_path, M)
                n_created += 1

        import gc; gc.collect()
        print(f"  [{i+1}/{len(obs_files)}] {f.name} — {n_created} snapshots crees")

    n_days = len(days)
    val_days   = [days[i] for i in range(4, n_days, 5)]
    test_days  = [days[i] for i in range(2, n_days, 5)]
    train_days = [d for d in days if d not in val_days and d not in test_days]

    split = {
        "train_days": train_days, "val_days": val_days, "test_days": test_days,
        "heures": list(hours), "nx": NX, "ny": NY, "nz": NZ,
        "z_levels": Z_LEVELS.tolist(),
        "n_input": N_INPUT_CH, "n_target": len(TARGET_COLS),
        "input_cols": input_norm_cols, "target_cols": TARGET_COLS,
        "day_col": DAY_COL, "hour_col": HOUR_COL,
        "land_use_map": {str(k): v for k, v in land_use_map.items()},
        "lu_mean": lu_mean, "lu_std": lu_std,
        "mean_dict": mean_dict, "std_dict": std_dict,
        "downsample": DOWNSAMPLE,
    }
    with open(TENSOR_DIR / "split.json", "w") as f:
        json.dump(split, f, indent=2)

    print(f"\n  Train : {len(train_days)} jours")
    print(f"  Val   : {len(val_days)} jours")
    print(f"  Test  : {len(test_days)} jours")
    print(f"\n✓ Pipeline 3D termine → {TENSOR_DIR}")

In [ ]:
with open(TENSOR_DIR / "split.json", "r") as f:
    split = json.load(f)

TRAIN_DAYS  = split["train_days"]
VAL_DAYS    = split["val_days"]
TEST_DAYS   = split["test_days"]
HEURES      = split["heures"]
NX          = split["nx"]
NY          = split["ny"]
NZ          = split["nz"]
Z_LEVELS    = np.array(split["z_levels"], dtype=np.float32)
N_INPUT     = split["n_input"]
N_TARGET    = split["n_target"]
TARGET_COLS = split["target_cols"]

# ordre de TARGET_COLS : u, v, w, rho, T, RH, p — la loss physique indexe ces canaux
IDX_U, IDX_V, IDX_W, IDX_RHO = 0, 1, 2, 3

norm_stats = pd.read_csv(OUT_DIR / "normalization_stats.csv", index_col=0)
MU_RHO  = float(norm_stats.loc["air_density_kg_m3", "mean"])
STD_RHO = float(norm_stats.loc["air_density_kg_m3", "std"])

N_STATIC_FEATURES   = 7   # z_terrain, z0, slope, aspect_sin, aspect_cos, x, y
N_LEVEL_FEATURES    = 2   # height_agl_norm, valid_mask (varient par niveau z)
N_TEMPORAL_FEATURES = 2   # hour_sin, hour_cos
N_INPUT_EFFECTIVE = N_INPUT + N_STATIC_FEATURES + N_LEVEL_FEATURES + N_TEMPORAL_FEATURES

print("Configuration FNO3D :\n")
print(f"  Train : {len(TRAIN_DAYS)} jours")
print(f"  Val   : {len(VAL_DAYS)} jours")
print(f"  Test  : {len(TEST_DAYS)} jours")
print(f"\n  Grille : {NX} x {NY} x {NZ} niveaux ASL {Z_LEVELS.tolist()}")
print(f"  N_INPUT_EFFECTIVE : {N_INPUT_EFFECTIVE}")
print(f"  N_TARGET          : {N_TARGET}  (ordre: {TARGET_COLS})")
print(f"\n  ρ physique : μ={MU_RHO:.4f} kg/m³  σ={STD_RHO:.4f} kg/m³")
print(f"\n✓ Config OK")

In [ ]:
class ANEODataset3D(Dataset):
    """
    Input 2D (une observation par colonne) leve en 3D par broadcast sur z, puis
    concatene avec les features statiques 3D (terrain, hauteur au-dessus du sol,
    masque de validite) et temporelles. Target et masque restent 3D natifs.
    """
    def __init__(self, days, heures, tensor_dir, static_npz, augment=False):
        self.tensor_dir = Path(tensor_dir)
        self.augment    = augment
        self.samples    = []

        for day in days:
            for h in heures:
                x_path = self.tensor_dir / f"day_{day:03d}_h{h:02d}_X.npy"
                y_path = self.tensor_dir / f"day_{day:03d}_h{h:02d}_Y.npy"
                m_path = self.tensor_dir / f"day_{day:03d}_h{h:02d}_M.npy"
                if x_path.exists() and y_path.exists() and m_path.exists():
                    self.samples.append((x_path, y_path, m_path, day, h))

        g = np.load(static_npz)
        static_features = g['static_features']     # (7, NY, NX)
        height_agl_norm = g['height_agl_norm']      # (NZ, NY, NX)
        valid_mask      = g['valid_mask']            # (NZ, NY, NX)

        self.static_3d = torch.from_numpy(
            np.repeat(static_features[:, None, :, :], NZ, axis=1)).float()   # (7, NZ, NY, NX)
        self.height_agl_3d = torch.from_numpy(height_agl_norm).float()[None]  # (1, NZ, NY, NX)
        self.valid_mask_3d = torch.from_numpy(valid_mask).float()[None]      # (1, NZ, NY, NX)

        print(f"  Dataset : {len(self.samples)} snapshots "
              f"({'aug ON' if augment else 'aug OFF'})")

    def __len__(self):
        return len(self.samples)

    def augment_sample(self, X, Y, M):
        X, Y = X.clone(), Y.clone()
        if torch.rand(1) > 0.5:
            scale = torch.FloatTensor(1).uniform_(0.8, 1.2).item()
            X[0:3] *= scale   # wind_u,v,w = 3 premiers canaux dynamiques (voir TARGET_COLS)
            Y[0:3] *= scale
        if torch.rand(1) > 0.5:
            X[:8] += torch.randn_like(X[:8]) * 0.02
        return X, Y, M

    def __getitem__(self, idx):
        x_path, y_path, m_path, day, h = self.samples[idx]

        X_2d = torch.from_numpy(np.load(x_path).astype(np.float32).transpose(2, 0, 1))       # (8, NY, NX)
        Y_3d = torch.from_numpy(np.load(y_path).astype(np.float32).transpose(3, 0, 1, 2))     # (7, NZ, NY, NX)
        M_3d = torch.from_numpy(np.load(m_path).astype(np.float32).transpose(2, 0, 1))[None]  # (1, NZ, NY, NX)

        if self.augment:
            X_2d, Y_3d, M_3d = self.augment_sample(X_2d, Y_3d, M_3d)

        X_3d = X_2d[:, None, :, :].expand(-1, NZ, -1, -1)   # leve 2D -> 3D (broadcast sur z)

        hour_sin = float(np.sin(2 * np.pi * h / 24))
        hour_cos = float(np.cos(2 * np.pi * h / 24))
        _, H, W = X_2d.shape
        sin_ch = torch.full((1, NZ, H, W), hour_sin, dtype=torch.float32)
        cos_ch = torch.full((1, NZ, H, W), hour_cos, dtype=torch.float32)

        X = torch.cat([X_3d, self.static_3d, self.height_agl_3d, self.valid_mask_3d,
                       sin_ch, cos_ch], dim=0)

        return X, Y_3d, M_3d, day, h


BATCH_SIZE = 1

train_dataset = ANEODataset3D(TRAIN_DAYS + VAL_DAYS, HEURES, TENSOR_DIR, GRID3D_PATH, augment=True)
val_dataset   = ANEODataset3D(VAL_DAYS, HEURES, TENSOR_DIR, GRID3D_PATH, augment=False)
test_dataset  = ANEODataset3D(TEST_DAYS, HEURES, TENSOR_DIR, GRID3D_PATH, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=1,          shuffle=False, num_workers=2)

print(f"Batch size    : {BATCH_SIZE}")
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

X_b, Y_b, M_b, _, _ = next(iter(train_loader))
print(f"\nX : {X_b.shape}  ← doit etre [1, {N_INPUT_EFFECTIVE}, {NZ}, {NY}, {NX}]")
print(f"Y : {Y_b.shape}  ← doit etre [1, {N_TARGET}, {NZ}, {NY}, {NX}]")
print(f"M : {M_b.shape}  ← doit etre [1, 1, {NZ}, {NY}, {NX}]")

assert X_b.shape[1] == N_INPUT_EFFECTIVE, \
    f"Channels incorrects: {X_b.shape[1]} vs {N_INPUT_EFFECTIVE}"

print(f"\n✓ Dataloaders 3D prets")

In [ ]:
class SpectralConv3d(nn.Module):
    """FFT3D tronquee sur (z, y, x) — x est l'axe reel (rfftn), z et y ont chacun
    les deux signes de frequence (4 coins), comme l'implementation 2D de fno2D.ipynb
    etendue d'un axe."""
    def __init__(self, in_channels, out_channels, modes_z, modes_y, modes_x):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.modes_z = modes_z
        self.modes_y = modes_y
        self.modes_x = modes_x
        scale = 1 / (in_channels * out_channels)
        shape = (in_channels, out_channels, modes_z, modes_y, modes_x)
        self.weights1 = nn.Parameter(scale * torch.rand(*shape, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(scale * torch.rand(*shape, dtype=torch.cfloat))
        self.weights3 = nn.Parameter(scale * torch.rand(*shape, dtype=torch.cfloat))
        self.weights4 = nn.Parameter(scale * torch.rand(*shape, dtype=torch.cfloat))

    def compl_mul3d(self, input, weights):
        return torch.einsum("bizyx,iozyx->bozyx", input, weights)

    def forward(self, x):
        B = x.shape[0]
        Z, Y, X = x.size(-3), x.size(-2), x.size(-1)
        x_ft = torch.fft.rfftn(x.float(), dim=[-3, -2, -1])
        out_ft = torch.zeros(
            B, self.out_channels, Z, Y, X // 2 + 1,
            dtype=torch.cfloat, device=x.device)

        mz, my, mx = self.modes_z, self.modes_y, self.modes_x
        out_ft[:, :, :mz, :my, :mx]  = self.compl_mul3d(x_ft[:, :, :mz, :my, :mx],  self.weights1)
        out_ft[:, :, -mz:, :my, :mx] = self.compl_mul3d(x_ft[:, :, -mz:, :my, :mx], self.weights2)
        out_ft[:, :, :mz, -my:, :mx] = self.compl_mul3d(x_ft[:, :, :mz, -my:, :mx], self.weights3)
        out_ft[:, :, -mz:, -my:, :mx] = self.compl_mul3d(x_ft[:, :, -mz:, -my:, :mx], self.weights4)

        return torch.fft.irfftn(out_ft, s=(Z, Y, X), dim=[-3, -2, -1])


class FNOBlock3d(nn.Module):
    def __init__(self, channels, modes_z, modes_y, modes_x, dropout=0.05):
        super().__init__()
        self.spectral = SpectralConv3d(channels, channels, modes_z, modes_y, modes_x)
        self.local    = nn.Conv3d(channels, channels, kernel_size=1)
        self.norm     = nn.InstanceNorm3d(channels, affine=True)
        self.dropout  = nn.Dropout3d(p=dropout)

    def forward(self, x):
        return F.gelu(self.dropout(
            self.norm(self.spectral(x) + self.local(x))))


class FNO3D(nn.Module):
    def __init__(self, in_channels, out_channels,
                 hidden_channels=48, n_blocks=4,
                 modes_z=3, modes_y=16, modes_x=16, dropout=0.05):
        super().__init__()
        self.lifting = nn.Sequential(
            nn.Conv3d(in_channels, hidden_channels, kernel_size=1),
            nn.GELU(),
        )
        self.blocks = nn.ModuleList([
            FNOBlock3d(hidden_channels, modes_z, modes_y, modes_x, dropout)
            for _ in range(n_blocks)
        ])
        self.skip_proj = nn.Conv3d(hidden_channels, hidden_channels, kernel_size=1)

        def make_head(h):
            return nn.Sequential(
                nn.Conv3d(h, h//2, kernel_size=1), nn.GELU(),
                nn.Conv3d(h//2, h//4, kernel_size=1), nn.GELU(),
                nn.Conv3d(h//4, 1,   kernel_size=1),
            )

        self.heads = nn.ModuleList([make_head(hidden_channels) for _ in range(out_channels)])

    def forward(self, x):
        x    = self.lifting(x)
        skip = self.skip_proj(x)
        for block in self.blocks:
            x = block(x)
        x = x + skip
        return torch.cat([head(x) for head in self.heads], dim=1)


model = FNO3D(
    in_channels     = N_INPUT_EFFECTIVE,
    out_channels    = N_TARGET,
    hidden_channels = 48,
    n_blocks        = 4,
    modes_z         = 3,
    modes_y         = 16,
    modes_x         = 16,
    dropout         = 0.05,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FNO3D Model :")
print(f"  in_channels     : {N_INPUT_EFFECTIVE}")
print(f"  out_channels    : {N_TARGET}  ({TARGET_COLS})")
print(f"  hidden_channels : 48")
print(f"  n_blocks        : 4")
print(f"  modes (z,y,x)   : 3, 16, 16")
print(f"  parametres      : {n_params:,}")
print(f"  device          : {device}")
print(f"\n✓ Modele FNO3D pret")

In [ ]:
class PhysicsConstrainedLoss3D(nn.Module):
    """
    L = L_data + lambda_mass*L_mass + lambda_bc*L_bc + lambda_smooth*L_smooth
    (meme forme que la specification section 4.5, etendue a un vrai volume 3D) :
      - L_data   : MSE uniquement aux voxels avec une vraie observation ANEO (obs_mask).
      - L_mass   : div(rho*v) = d(rho*u)/dx + d(rho*v)/dy + d(rho*w)/dz complet, avec
                   correction de pente terrain sur (x,y) et vraie difference finie en z
                   (niveaux ASL non-uniformement espaces) ; masque aux voxels souterrains.
      - L_bc     : w=0 au premier niveau valide (sol reel) de chaque colonne, + gradient
                   vertical faible de (u,v) au niveau superieur (1000m, cf. section 4.4).
      - L_smooth : continuite spatiale (x,y), masquee aux voxels souterrains.
    """
    def __init__(self, lambda_mass=0.3, lambda_bc=0.05, lambda_smooth=0.01,
                 dx=100.0, mu_rho=1.154, std_rho=0.021,
                 static_npz=None, z_levels=None):
        super().__init__()
        self.lambda_mass   = lambda_mass
        self.lambda_bc     = lambda_bc
        self.lambda_smooth = lambda_smooth
        self.dx = dx
        self.z_levels = [float(z) for z in z_levels]   # niveaux ASL (m), espacement non-uniforme

        self.register_buffer('mu_rho',  torch.tensor(mu_rho,  dtype=torch.float32))
        self.register_buffer('std_rho', torch.tensor(std_rho, dtype=torch.float32))

        g = np.load(static_npz)
        z_terrain  = g['z_terrain']    # (NY, NX)
        valid_mask = g['valid_mask']   # (NZ, NY, NX)

        self.register_buffer('z_terrain',  torch.from_numpy(z_terrain).float()[None, None, None])
        self.register_buffer('valid_mask', torch.from_numpy(valid_mask).float()[None, None])

        bottom_idx = valid_mask.argmax(axis=0)   # (NY, NX) — 1er niveau valide (au-dessus du sol)
        self.register_buffer('bottom_idx', torch.from_numpy(bottom_idx).long()[None, None])

    def d_dz(self, field):
        """field: (B,1,NZ,NY,NX) -> d(field)/dz, differences finies, espacement non-uniforme."""
        z = self.z_levels
        out = torch.empty_like(field)
        out[:, :, 0]  = (field[:, :, 1]  - field[:, :, 0])  / (z[1]  - z[0])
        out[:, :, -1] = (field[:, :, -1] - field[:, :, -2]) / (z[-1] - z[-2])
        for i in range(1, field.shape[2] - 1):
            out[:, :, i] = (field[:, :, i+1] - field[:, :, i-1]) / (z[i+1] - z[i-1])
        return out

    def forward(self, pred, target, obs_mask, phys_weight=1.0):
        L_data = ((pred - target) ** 2 * obs_mask).sum() / (obs_mask.sum() + 1e-8)

        u_pred = pred[:, 0:1]
        v_pred = pred[:, 1:2]
        w_pred = pred[:, 2:3]
        rho_phys = pred[:, 3:4] * self.std_rho + self.mu_rho   # densite predite, denormalisee

        rho_u = rho_phys * u_pred
        rho_v = rho_phys * v_pred
        rho_w = rho_phys * w_pred

        # pas horizontal corrige par la pente du terrain (identique a la correction fno2D)
        dzt_dx = self.z_terrain[..., 2:] - self.z_terrain[..., :-2]
        dzt_dy = self.z_terrain[..., 2:, :] - self.z_terrain[..., :-2, :]
        ds_x = torch.sqrt((2 * self.dx) ** 2 + dzt_dx ** 2)
        ds_y = torch.sqrt((2 * self.dx) ** 2 + dzt_dy ** 2)

        drhou_dx = (rho_u[..., 2:] - rho_u[..., :-2]) / ds_x
        drhov_dy = (rho_v[..., 2:, :] - rho_v[..., :-2, :]) / ds_y
        min_y = min(drhou_dx.shape[-2], drhov_dy.shape[-2])
        min_x = min(drhou_dx.shape[-1], drhov_dy.shape[-1])

        drhow_dz = self.d_dz(rho_w)   # vraie derivee verticale, pas d'approximation 1-niveau

        div = (drhou_dx[..., :min_y, :min_x] +
               drhov_dy[..., :min_y, :min_x] +
               drhow_dz[..., :min_y, :min_x])
        mask_c = self.valid_mask[..., :min_y, :min_x]
        L_mass = ((div ** 2) * mask_c).sum() / (mask_c.sum() + 1e-8)

        grad_u_x = u_pred[..., 1:]   - u_pred[..., :-1]
        grad_u_y = u_pred[..., 1:, :] - u_pred[..., :-1, :]
        grad_v_x = v_pred[..., 1:]   - v_pred[..., :-1]
        grad_v_y = v_pred[..., 1:, :] - v_pred[..., :-1, :]
        mx = self.valid_mask[..., 1:]
        my = self.valid_mask[..., 1:, :]
        L_smooth = (
            ((grad_u_x ** 2) * mx).sum() / (mx.sum() + 1e-8) +
            ((grad_v_x ** 2) * mx).sum() / (mx.sum() + 1e-8) +
            ((grad_u_y ** 2) * my).sum() / (my.sum() + 1e-8) +
            ((grad_v_y ** 2) * my).sum() / (my.sum() + 1e-8)
        )

        # w=0 au sol reel de chaque colonne (pas au niveau ASL le plus bas globalement)
        B, _, _, H, W = pred.shape
        idx = self.bottom_idx.expand(B, 1, H, W).unsqueeze(2)
        w_ground = torch.gather(w_pred, dim=2, index=idx).squeeze(2)
        L_bc_ground = (w_ground ** 2).mean()

        # gradient vertical faible pres du sommet (1000m), cf. specification section 4.4
        dz_top = self.z_levels[-1] - self.z_levels[-2]
        grad_top_u = (u_pred[:, :, -1] - u_pred[:, :, -2]) / dz_top
        grad_top_v = (v_pred[:, :, -1] - v_pred[:, :, -2]) / dz_top
        L_bc = L_bc_ground + (grad_top_u ** 2).mean() + (grad_top_v ** 2).mean()

        phys_term = (self.lambda_mass   * L_mass +
                     self.lambda_bc     * L_bc +
                     self.lambda_smooth * L_smooth)
        L_total = L_data + phys_weight * phys_term
        return L_total, L_data, L_mass, L_bc, L_smooth


criterion = PhysicsConstrainedLoss3D(
    lambda_mass=0.3, lambda_bc=0.05, lambda_smooth=0.01,
    dx=50.0 * DOWNSAMPLE, mu_rho=MU_RHO, std_rho=STD_RHO,
    static_npz=GRID3D_PATH, z_levels=Z_LEVELS,
).to(device)
print(f"✓ Physics loss 3D prete (masse 3D complete + pente terrain + BC sol/altitude)")
print(f"  dx effectif : {50.0*DOWNSAMPLE:.1f}m (grille sous-echantillonnee x{DOWNSAMPLE})")

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = 3e-4,
    weight_decay = 1e-4,
    betas        = (0.9, 0.999),
)

N_EPOCHS      = 150
WARMUP_EPOCHS = 15
EARLY_STOP    = 30
ACCUM_STEPS   = 4

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-6)

scaler = GradScaler('cuda')


def physics_warmup(epoch, warmup_epochs):
    return min(1.0, epoch / max(warmup_epochs, 1))


print(f"  N_EPOCHS      : {N_EPOCHS}")
print(f"  WARMUP_EPOCHS : {WARMUP_EPOCHS}")
print(f"  EARLY_STOP    : {EARLY_STOP}")
print(f"  ACCUM_STEPS   : {ACCUM_STEPS}")
print(f"\n✓ Optimizer pret")

In [ ]:
def train_epoch(epoch):
    model.train()
    phys_w = physics_warmup(epoch, WARMUP_EPOCHS)
    tot_loss = tot_data = tot_mass = tot_bc = tot_smooth = 0.0
    n = 0
    optimizer.zero_grad()
    for step, (X_b, Y_b, M_b, day, hour) in enumerate(train_loader):
        X_b, Y_b, M_b = X_b.to(device), Y_b.to(device), M_b.to(device)
        with autocast('cuda'):
            pred = model(X_b.float())
            loss, l_data, l_mass, l_bc, l_smooth = criterion(pred, Y_b, M_b, phys_w)
            loss = loss / ACCUM_STEPS
        scaler.scale(loss).backward()
        if (step + 1) % ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        tot_loss   += loss.item() * ACCUM_STEPS
        tot_data   += l_data.item()
        tot_mass   += l_mass.item()
        tot_bc     += l_bc.item()
        tot_smooth += l_smooth.item()
        n += 1
    return tot_loss/n, tot_data/n, tot_mass/n, tot_bc/n, tot_smooth/n


def validate_epoch(epoch):
    model.eval()
    tot_loss = tot_data = tot_mass = tot_bc = 0.0
    n = 0
    all_r2 = []
    with torch.no_grad():
        for X_b, Y_b, M_b, day, hour in val_loader:
            X_b, Y_b, M_b = X_b.to(device), Y_b.to(device), M_b.to(device)
            with autocast('cuda'):
                pred = model(X_b.float())
                loss, l_data, l_mass, l_bc, l_smooth = criterion(pred, Y_b, M_b, 1.0)
            tot_loss += loss.item(); tot_data += l_data.item()
            tot_mass += l_mass.item(); tot_bc += l_bc.item()

            obs = M_b.bool().expand_as(pred)
            if obs.sum() > 0:
                p = pred[obs].float(); t = Y_b[obs].float()
                ss_res = ((p - t) ** 2).sum()
                ss_tot = ((t - t.mean()) ** 2).sum() + 1e-9
                all_r2.append((1 - ss_res / ss_tot).item())
            n += 1
    r2 = float(np.mean(all_r2)) if all_r2 else float('nan')
    return tot_loss/n, tot_data/n, tot_mass/n, tot_bc/n, r2


print("✓ train_epoch / validate_epoch definis")

In [ ]:
history = {k: [] for k in [
    'train_loss', 'val_loss', 'r2', 'lr', 'phys_weight',
    'train_l_data', 'train_l_mass', 'train_l_bc', 'train_l_smooth',
    'val_l_data', 'val_l_mass', 'val_l_bc',
]}

best_val_loss    = float('inf')
best_epoch       = 0
best_r2          = -np.inf
patience_counter = 0

print("=" * 100)
print("STARTING FNO3D TRAINING")
print("=" * 100)

try:
    for epoch in range(N_EPOCHS):
        tr_loss, tr_data, tr_mass, tr_bc, tr_smooth = train_epoch(epoch)
        va_loss, va_data, va_mass, va_bc, r2        = validate_epoch(epoch)
        scheduler.step()

        lr     = optimizer.param_groups[0]['lr']
        phys_w = physics_warmup(epoch, WARMUP_EPOCHS)

        history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
        history['r2'].append(r2); history['lr'].append(lr); history['phys_weight'].append(phys_w)
        history['train_l_data'].append(tr_data); history['train_l_mass'].append(tr_mass)
        history['train_l_bc'].append(tr_bc); history['train_l_smooth'].append(tr_smooth)
        history['val_l_data'].append(va_data); history['val_l_mass'].append(va_mass)
        history['val_l_bc'].append(va_bc)

        print(f"{epoch+1:>4} | train={tr_loss:.5f} | val={va_loss:.5f} | R²={r2:.4f} | "
              f"phi={phys_w:.2f} | lr={lr:.2e} | ", end="")

        if va_loss < best_val_loss:
            best_val_loss, best_epoch, best_r2 = va_loss, epoch + 1, r2
            patience_counter = 0
            torch.save({
                'epoch'               : epoch + 1,
                'model_state_dict'    : model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict'   : scaler.state_dict(),
                'val_loss'            : va_loss,
                'r2'                  : r2,
                'history'             : history,
                'val_l_data'          : va_data,
                'val_l_mass'          : va_mass,
                'val_l_bc'            : va_bc,
            }, MODEL_DIR / 'best_fno3d_model.pth')
            # sauvegarde Drive immediate — lecon tiree du checkpoint FNO2D perdu (2026-07-17)
            shutil.copy2(MODEL_DIR / 'best_fno3d_model.pth',
                         DRIVE_MODEL_DIR / 'best_fno3d_model.pth')
            print("✓ BEST (saved to Drive)")
        else:
            patience_counter += 1
            print(f"(patience {patience_counter}/{EARLY_STOP})"
                  if patience_counter % 5 == 0 else "")

        if patience_counter >= EARLY_STOP:
            print(f"\n⚠ Early stopping a l'epoque {epoch+1}")
            break

except KeyboardInterrupt:
    print("\n⚠ Entrainement interrompu")
except Exception as e:
    print(f"\n✗ Echec: {type(e).__name__}: {e}")
    import traceback; traceback.print_exc()

print("\n" + "=" * 100)
print(f"TERMINE — meilleure epoque {best_epoch}  val_loss={best_val_loss:.6f}  R²={best_r2:.4f}")
print("=" * 100)

pd.DataFrame(history).to_csv(OUT_DIR / 'fno3d_training_history.csv', index=False)
shutil.copy2(OUT_DIR / 'fno3d_training_history.csv',
             DRIVE_MODEL_DIR / 'fno3d_training_history.csv')
print(f"✓ Checkpoint + historique copies vers Drive ({DRIVE_MODEL_DIR})")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss totale'); axes[0].set_xlabel('epoque'); axes[0].legend()

axes[1].plot(history['train_l_data'], label='L_data')
axes[1].plot(history['train_l_mass'], label='L_mass')
axes[1].plot(history['train_l_bc'], label='L_bc')
axes[1].plot(history['train_l_smooth'], label='L_smooth')
axes[1].set_yscale('log')
axes[1].set_title('Composantes de la loss (train)'); axes[1].set_xlabel('epoque'); axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'loss_components_3d.png', dpi=150, bbox_inches='tight')
print(f"✓ Figure sauvegardee → {FIG_DIR / 'loss_components_3d.png'}")

In [ ]:
ckpt = torch.load(MODEL_DIR / "best_fno3d_model.pth", map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Charge epoque {ckpt['epoch']}  val_loss={ckpt['val_loss']:.6f}  R²={ckpt['r2']:.4f}")

mean_dict = split["mean_dict"]
std_dict  = split["std_dict"]

results = []
with torch.no_grad():
    for X_b, Y_b, M_b, day, hour in test_loader:
        day, hour = day.item(), hour.item()
        X_b, Y_b, M_b = X_b.to(device), Y_b.to(device), M_b.to(device)
        pred = model(X_b.float())

        mask_bool = M_b[0, 0] > 0.5   # (NZ, NY, NX)

        for zi, zlevel in enumerate(Z_LEVELS):
            level_mask = mask_bool[zi]
            n_obs = int(level_mask.sum().item())
            if n_obs == 0:
                continue
            row = dict(day=day, hour=hour, z_level=float(zlevel), n_obs=n_obs)
            for ci, col in enumerate(TARGET_COLS):
                p = pred[0, ci, zi][level_mask].float().cpu().numpy()
                t = Y_b[0, ci, zi][level_mask].float().cpu().numpy()
                mu, std = mean_dict[col], std_dict[col]
                p_phys, t_phys = p * std + mu, t * std + mu
                rmse = float(np.sqrt(np.mean((p_phys - t_phys) ** 2)))
                mae  = float(np.mean(np.abs(p_phys - t_phys)))
                ss_res = np.sum((p_phys - t_phys) ** 2)
                ss_tot = np.sum((t_phys - t_phys.mean()) ** 2) + 1e-9
                r2 = float(1 - ss_res / ss_tot)
                row[f'rmse_{col}'] = rmse
                row[f'mae_{col}']  = mae
                row[f'r2_{col}']   = r2
            results.append(row)
        print(f"  J{day:3d} h{hour:02d} — "
              f"{int(mask_bool.sum().item())} voxels observes evalues")

df_r = pd.DataFrame(results)
df_r.to_csv(OUT_DIR / 'fno3d_evaluation_results.csv', index=False)

print(f"\n{'='*100}")
print("METRIQUES PAR NIVEAU ASL (moyenne sur le test set, uniquement aux voxels observes) :")
print(f"{'='*100}")
for zlevel in Z_LEVELS:
    sub = df_r[df_r['z_level'] == zlevel]
    if len(sub) == 0:
        print(f"  {zlevel:>6.0f}m : aucune observation dans le test set")
        continue
    print(f"  {zlevel:>6.0f}m ASL (n={sub['n_obs'].sum():.0f} voxels) : "
          f"RMSE_u={sub['rmse_wind_u_m_s'].mean():.4f}  "
          f"RMSE_v={sub['rmse_wind_v_m_s'].mean():.4f}  "
          f"R²_u={sub['r2_wind_u_m_s'].mean():.4f}  "
          f"R²_v={sub['r2_wind_v_m_s'].mean():.4f}")
print(f"{'='*100}")
print("\n⚠ Ces metriques ne valident QUE les voxels avec une vraie observation ANEO.")
print("  Le reste du volume 3D (interpole/extrapole) n'a pas de verite-terrain pour le valider —")
print("  seule la coherence physique (L_mass, L_bc) le contraint. Voir tukhnanutha.md section 3.")
print(f"\n✓ Resultats sauvegardes → {OUT_DIR / 'fno3d_evaluation_results.csv'}")

In [ ]:
# Profil vertical de vitesse moyenne predite (une colonne representative), pour un
# snapshot de test — verifie visuellement que le profil croit avec l'altitude
# (coherent avec la loi log/puissance attendue en couche limite atmospherique).
X_b, Y_b, M_b, day, hour = next(iter(test_loader))
day, hour = day.item(), hour.item()
X_b = X_b.to(device)

with torch.no_grad():
    pred = model(X_b.float())[0].float().cpu().numpy()   # (7, NZ, NY, NX)

valid_np = criterion.valid_mask[0, 0].cpu().numpy()   # (NZ, NY, NX)
ws = np.sqrt(pred[0]**2 + pred[1]**2)                 # vitesse horizontale (NZ, NY, NX)

mean_ws_per_level = [
    float(ws[zi][valid_np[zi] > 0.5].mean()) if (valid_np[zi] > 0.5).any() else np.nan
    for zi in range(NZ)
]

plt.figure(figsize=(5, 6))
plt.plot(mean_ws_per_level, Z_LEVELS, marker='o')
plt.xlabel("Vitesse horizontale moyenne predite (m/s)")
plt.ylabel("Altitude ASL (m)")
plt.title(f"Profil vertical predit — jour {day} h{hour:02d}")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / f'vertical_profile_day{day:03d}_h{hour:02d}.png', dpi=150, bbox_inches='tight')
print(f"✓ Figure sauvegardee → {FIG_DIR}/vertical_profile_day{day:03d}_h{hour:02d}.png")
print(f"  Profil (m ASL → m/s) : "
      + ", ".join(f"{z:.0f}m={v:.2f}" for z, v in zip(Z_LEVELS, mean_ws_per_level)))